# Feature Engineering & Data Analysis

**Continuing from:** `exploring_cleaning_data.ipynb` — this notebook picks up once the data is *clean*, and turns it into something that can actually answer business questions.

## Why this phase exists

Cleaning makes data **correct** — no missing values, no duplicates, the right dtypes. That's necessary, but it's not the same as data being **useful**. A clean `Order Date` column doesn't tell you whether Mondays sell more than Fridays; a clean `Sales`/`Profit` pair doesn't tell you which customers are actually worth the most. Feature engineering is the step where we *derive* new columns that carry that signal explicitly, and data analysis is the step where we use those columns (plus the originals) to actually answer questions.

Skipping straight from cleaning to charts is a common shortcut — and it produces charts that answer questions nobody asked, because the underlying features were never designed around a question in the first place. This notebook keeps the two steps in order: **know the question → build the feature that answers it → analyze it.**

## The Generic Workflow

This is the same repeatable process regardless of the dataset — the checklist we'll apply to the Superstore data in Part 2 below.

0. **Setup & Recap** — load the cleaned data, confirm it survived the handoff intact.
1. **Define the Analysis Questions** — write down what we're trying to answer *before* building features, so every feature has a reason to exist.
2. **Feature Engineering — Derived Columns** — transform existing columns (dates, numeric ratios, categorical cleanup) into new per-row signal.
3. **Feature Engineering — Aggregated Features** — roll row-level data up into entity-level features (per customer, per product, etc.).
4. **Feature Validation** — sanity-check every new feature immediately: nulls introduced by the transform, implausible ranges, divide-by-zero.
5. **Exploratory Data Analysis** — univariate → bivariate → multivariate, in that order.
6. **Answering the Questions** — go back to Step 1 and directly resolve each question with a specific table or aggregation.
7. **Insight Synthesis** — summarize findings in plain language, including caveats inherited from cleaning decisions.
8. **Handoff Prep** — name and preserve the tables/features the next stage (visualization) will need.

## Part 2 — Applying the Workflow to the Superstore Dataset

From here on, every section is Part 2: the same nine steps, made concrete for `Sample-Superstore2019.csv`.

> **Note on notebooks and kernels:** this notebook does **not** share memory with `exploring_cleaning_data.ipynb` — each `.ipynb` file runs its own kernel. Step 0 below re-loads the raw CSV and re-applies the cleaning decisions already validated there, rather than assuming `df` already exists. This keeps the notebook runnable on its own.

### Step 0 — Setup & Recap

We re-establish the clean baseline before building anything new. Nothing new is decided here, we're just reproducing what the cleaning notebook already validated:

- Drop `Unnamed: 0` and `Row ID` (load artifacts, not data)
- Convert `Order Date` / `Ship Date` to real `datetime`
- Fill the 11 missing `Postal Code` values (all Burlington, VT) with `05401`
- Convert low-cardinality text columns to `category`
- Drop the single true exact-duplicate row

One deliberate omission: the 7 `Order ID` + `Product ID` pairs that share a combination but have different `Quantity`/`Sales` are **not** collapsed. Investigation in the cleaning notebook showed each pair has matching unit price and unit profit — they're legitimate separate order lines, not data-entry errors, and dropping either row would silently discard real revenue.

In [1]:
# Importing the proper and required data packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pyarrow

In [2]:
# Reading the clean data and make a copy of the original file to be the single source of truth
df = pd.read_parquet("Sample_Superstore_2019_Clean.parquet", engine="pyarrow")
df.shape

(9993, 20)

In [3]:
# Creating the copy of the original DataFrame
df_clean = df.copy()
df_clean.info(memory_usage='deep')

<class 'pandas.DataFrame'>
RangeIndex: 9993 entries, 0 to 9992
Data columns (total 20 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Order ID        9993 non-null   str           
 1   Order Date      9993 non-null   datetime64[us]
 2   Ship Date       9993 non-null   datetime64[us]
 3   Ship Mode       9993 non-null   category      
 4   Customer ID     9993 non-null   str           
 5   Customer Name   9993 non-null   str           
 6   Segment         9993 non-null   category      
 7   Country/Region  9993 non-null   category      
 8   City            9993 non-null   str           
 9   State           9993 non-null   category      
 10  Postal Code     9993 non-null   str           
 11  Region          9993 non-null   category      
 12  Product ID      9993 non-null   str           
 13  Category        9993 non-null   category      
 14  Sub-Category    9993 non-null   category      
 15  Product Name   

### Step 1 — Define the Analysis Questions

These are the questions the rest of this notebook is built to answer. Every feature engineered in Steps 2–3 exists to serve at least one of these:

1. Which `State` generates the most total `Profit`, and which generates the biggest total loss?
2. Is there a relationship between `Discount` and `Profit`?
3. Does `Ship Mode` relate to profitability or shipping time?
4. Is there seasonality in `Sales` — by month, or by day of week?
5. Who are the most valuable customers, and how would we segment them?

Step 6 will come back to this exact list and answer each one directly.

### Step 2 — Feature Engineering: Derived Columns

Two families of per-row features, built directly from existing columns:

- **Date-based**, from `Order Date` / `Ship Date`: how long shipping took, and what calendar pattern the order falls into (weekday, month, quarter, year, weekend flag) — needed to test the seasonality question (Q4).
- **Numeric transforms**, from `Sales` / `Quantity` / `Discount`: a per-unit price, a profit-margin ratio, and a discount bucket — needed to test the discount question (Q2).

In [4]:
# Date-based features -- needed for the seasonality question (Q4)
df_clean["Shipping Days"] = (df_clean["Ship Date"] - df_clean["Order Date"]).dt.days
df_clean["Order Weekday"] = df_clean["Order Date"].dt.day_name()
df_clean["Order Month"] = df_clean["Order Date"].dt.month
df_clean["Order Quarter"] = df_clean["Order Date"].dt.quarter
df_clean["Order Year"] = df_clean["Order Date"].dt.year
df_clean["Is Weekend"] = df_clean["Order Date"].dt.dayofweek >= 5

df_clean[["Order Date", "Ship Date", "Shipping Days", "Order Weekday", "Is Weekend"]].head()

,Order Date,Ship Date,Shipping Days,Order Weekday,Is Weekend
0,2018-11-08,2018-11-11,3,Thursday,False
1,2018-11-08,2018-11-11,3,Thursday,False
2,2018-06-12,2018-06-16,4,Tuesday,False
3,2017-10-11,2017-10-18,7,Wednesday,False
4,2017-10-11,2017-10-18,7,Wednesday,False


In [5]:
# Numeric transforms -- needed for the discount question (Q2)
df_clean["Unit Price"] = df_clean["Sales"] / df_clean["Quantity"]
df_clean["Profit Margin"] = df_clean["Profit"] / df_clean["Sales"]

# Discount buckets: 0 = no discount, then Low/Medium/High bands.
# Bins chosen from the actual observed range (0.0 to 0.8).
df_clean["Discount Bucket"] = pd.cut(
    df_clean["Discount"],
    bins=[-0.01, 0, 0.2, 0.4, 1.0],
    labels=["None", "Low", "Medium", "High"],
)

df_clean[["Sales", "Quantity", "Unit Price", "Profit", "Profit Margin", "Discount", "Discount Bucket"]].head()

,Sales,Quantity,Unit Price,Profit,Profit Margin,Discount,Discount Bucket
0,261.9600,2,130.9800,41.9136,0.1600,0.00,None
1,731.9400,3,243.9800,219.5820,0.3000,0.00,None
2,14.6200,2,7.3100,6.8714,0.4700,0.00,None
3,957.5775,5,191.5155,-383.0310,-0.4000,0.45,High
4,22.3680,2,11.1840,2.5164,0.1125,0.20,Low


### Step 3 — Feature Engineering: Aggregated Features

Row-level features answer row-level questions. To answer "who are our most valuable customers?" (Q5) we need to roll individual order lines up to one row per entity:

- **Customer-level**: how often they order, how much they spend, how long they've been a customer
- **Product-level**: how much of each product sells, and at what average discount
- **RFM** (Recency / Frequency / Monetary): the classic customer-value framework, built directly from the two aggregates above plus `Order Date`

In [6]:
# One row per customer: order count, spend, average order value, tenure
customer_features = df_clean.groupby("Customer ID", observed=True).agg(
    order_count=("Order ID", "nunique"),
    total_spend=("Sales", "sum"),
    total_profit=("Profit", "sum"),
    avg_order_value=("Sales", "mean"),
    first_order=("Order Date", "min"),
    last_order=("Order Date", "max"),
)

# Tenure = span between a customer's first and last order in this dataset
customer_features["Customer Tenure Days"] = (
    customer_features["last_order"] - customer_features["first_order"]
).dt.days

customer_features.sort_values("total_spend", ascending=False).head()

,order_count,total_spend,total_profit,avg_order_value,first_order,last_order,Customer Tenure Days
Customer ID,,,,,,,
Sm-20320,5,25043.050,-1980.7393,1669.536667,2016-03-18,2019-10-12,1303
Tc-20980,5,19052.218,8981.3239,1587.684833,2016-11-07,2018-11-26,749
Rb-19360,6,15117.339,6976.0959,839.852167,2018-04-01,2019-09-25,542
Ta-21385,4,14595.620,4703.7883,1459.562000,2016-09-12,2019-10-22,1135
Ab-10105,10,14473.571,5444.8055,723.678550,2016-12-20,2019-11-19,1064


In [7]:
# One row per product: units sold, revenue, average discount given
product_features = df_clean.groupby("Product ID", observed=True).agg(
    total_qty_sold=("Quantity", "sum"),
    total_revenue=("Sales", "sum"),
    avg_discount=("Discount", "mean"),
).sort_values("total_revenue", ascending=False)

product_features.head()

,total_qty_sold,total_revenue,avg_discount
Product ID,,,
Tec-Co-10004722,20,61599.824,0.12
Off-Bi-10003527,31,27453.384,0.24
Tec-Ma-10002412,6,22638.480,0.50
Fur-Ch-10002024,39,21870.576,0.20
Off-Bi-10001359,37,19823.479,0.30


In [8]:
# RFM: Recency (days since last order), Frequency (distinct orders),
# Monetary (total Sales). Recency is measured against the day after the
# last order in the whole dataset, since we have no "today" to measure from.
snapshot_date = df_clean["Order Date"].max() + pd.Timedelta(days=1)

rfm = df_clean.groupby("Customer ID", observed=True).agg(
    Recency=("Order Date", lambda x: (snapshot_date - x.max()).days),
    Frequency=("Order ID", "nunique"),
    Monetary=("Sales", "sum"),
)

rfm.sort_values("Monetary", ascending=False).head()

,Recency,Frequency,Monetary
Customer ID,,,
Sm-20320,80,5,25043.050
Tc-20980,400,5,19052.218
Rb-19360,97,6,15117.339
Ta-21385,70,4,14595.620
Ab-10105,42,10,14473.571


### Step 4 — Feature Validation

New features can silently break things the raw columns never would: a ratio can divide by zero, a date subtraction can go negative if the source dates were wrong, a bin can leave rows unlabeled. Check every new feature once, right after creating it.

In [9]:
# Divide-by-zero / infinite values in the ratio features
print("Infinite Unit Price values:", np.isinf(df_clean["Unit Price"]).sum())
print("Infinite Profit Margin values:", np.isinf(df_clean["Profit Margin"]).sum())

# Shipping Days should never be negative -- that would mean a product
# shipped before it was ordered
print("Shipping Days range:", df_clean["Shipping Days"].min(), "to", df_clean["Shipping Days"].max())
print("Negative Shipping Days:", (df_clean["Shipping Days"] < 0).sum())

# Every row should have landed in exactly one Discount Bucket
print("Unlabeled Discount Bucket rows:", df_clean["Discount Bucket"].isnull().sum())

Infinite Unit Price values: 0
Infinite Profit Margin values: 0
Shipping Days range: 0 to 7
Negative Shipping Days: 0
Unlabeled Discount Bucket rows: 0


### Step 5 — Exploratory Data Analysis

Univariate first (what does one feature look like on its own), then bivariate (how does it relate to profit), then multivariate (multiple dimensions at once).

In [10]:
# Univariate: distribution of the new per-row features
print(df_clean["Discount Bucket"].value_counts())
print()
print(df_clean["Shipping Days"].describe())
print()
print(df_clean["Profit Margin"].describe())

Discount Bucket
None      4798
Low       3803
High       933
Medium     459
Name: count, dtype: int64

count    9993.000000
mean        3.958071
std         1.748024
min         0.000000
25%         3.000000
50%         4.000000
75%         5.000000
max         7.000000
Name: Shipping Days, dtype: float64

count    9993.000000
mean        0.120330
std         0.466775
min        -2.750000
25%         0.075000
50%         0.270000
75%         0.362500
max         0.500000
Name: Profit Margin, dtype: float64


In [11]:
# Bivariate: Profit against Discount Bucket, Sales against calendar features
print(df_clean.groupby("Discount Bucket", observed=True)["Profit"].mean())
print()
print(df_clean.groupby("Order Month", observed=True)["Sales"].sum())

Discount Bucket
None       66.900292
Low        26.501571
Medium    -78.007422
High     -106.708028
Name: Profit, dtype: float64

Order Month
1      94924.8356
2      59751.2514
3     205005.4888
4     137480.7566
5     155028.8117
6     152718.6793
7     147238.0970
8     159044.0630
9     307649.9457
10    200322.9847
11    352461.0710
12    325293.5035
Name: Sales, dtype: float64


**Multivariate**: once two dimensions aren't enough, `.groupby()` on multiple columns and `.pivot_table()` spread one categorical column across the columns of the result — closer to the shape a report or chart actually needs.

- `.groupby("column")` splits the DataFrame into groups sharing the same value; chaining an aggregation (`.sum()`, `.mean()`, `.agg()`) collapses each group into one row.
- `.pivot_table()` is the more flexible version — it lets a second categorical column spread across the result's columns.
- `.sort_values()` orders the result so the most/least interesting rows are easy to spot.

In [12]:
# Total profit and sales per Region, sorted from most to least profitable
region_summary = df_clean.groupby("Region", observed=True)[["Sales", "Profit"]].sum().sort_values("Profit", ascending=False)
region_summary

,Sales,Profit
Region,,
West,725457.8245,108418.4489
East,678499.8680,91534.8388
South,391721.9050,46749.4303
Central,501239.8908,39706.3625


In [13]:
# Multiple aggregations at once with .agg()
category_summary = df_clean.groupby("Category", observed=True).agg(
    total_sales=("Sales", "sum"),
    avg_profit=("Profit", "mean"),
    orders=("Order ID", "count"),
)
category_summary

,total_sales,avg_profit,orders
Category,,,
Furniture,741718.4233,8.709119,2120
Office Supplies,719047.0320,20.327050,6026
Technology,836154.0330,78.752002,1847


In [14]:
# pivot_table: average Sales by Region (rows) x Category (columns)
pivot = pd.pivot_table(
    df_clean,
    values="Sales",
    index="Region",
    columns="Category",
    aggfunc="mean",
    observed=True,
)
pivot

Category,Furniture,Office Supplies,Technology
Region,,,
Central,340.534644,117.458801,405.753124
East,346.683053,120.044425,495.278469
South,353.309289,126.282727,507.753952
West,357.302325,116.422377,420.687533


### Step 6 — Answering the Questions

Back to the five questions from Step 1 — each one resolved directly with the features built above.

In [15]:
# Q1: which State generates the most Profit, and which the biggest loss?
state_profit = df_clean.groupby("State", observed=True)["Profit"].sum().sort_values(ascending=False)
print("Highest-profit state:", state_profit.idxmax(), "->", round(state_profit.max(), 2))
print("Biggest-loss state:", state_profit.idxmin(), "->", round(state_profit.min(), 2))

Highest-profit state: California -> 76381.39
Biggest-loss state: Texas -> -25729.36


In [16]:
# Q2: relationship between Discount and Profit, using the bucket built in Step 2
discount_profit = df_clean.groupby("Discount Bucket", observed=True)["Profit"].mean()
discount_profit

Discount Bucket
None       66.900292
Low        26.501571
Medium    -78.007422
High     -106.708028
Name: Profit, dtype: float64

In [17]:
# Q3: does Ship Mode relate to profitability or shipping time?
ship_mode_summary = df_clean.groupby("Ship Mode", observed=True).agg(
    avg_profit=("Profit", "mean"),
    avg_shipping_days=("Shipping Days", "mean"),
)
ship_mode_summary

,avg_profit,avg_shipping_days
Ship Mode,,
First Class,31.839948,2.182705
Same Day,29.266591,0.044199
Second Class,29.535545,3.237532
Standard Class,27.501399,5.006704


In [18]:
# Q4: seasonality in Sales -- by month and by weekday
print(df_clean.groupby("Order Month", observed=True)["Sales"].sum())
print()
print(df_clean.groupby("Order Weekday", observed=True)["Sales"].sum().sort_values(ascending=False))

Order Month
1      94924.8356
2      59751.2514
3     205005.4888
4     137480.7566
5     155028.8117
6     152718.6793
7     147238.0970
8     159044.0630
9     307649.9457
10    200322.9847
11    352461.0710
12    325293.5035
Name: Sales, dtype: float64

Order Weekday
Monday       396014.2259
Wednesday    392859.0138
Tuesday      373021.0407
Sunday       341391.5781
Thursday     298677.7432
Saturday     287957.2686
Friday       206998.6180
Name: Sales, dtype: float64


In [19]:
# Q5: most valuable customers, using the RFM features from Step 3
rfm.sort_values("Monetary", ascending=False).head()

,Recency,Frequency,Monetary
Customer ID,,,
Sm-20320,80,5,25043.050
Tc-20980,400,5,19052.218
Rb-19360,97,6,15117.339
Ta-21385,70,4,14595.620
Ab-10105,42,10,14473.571


### Try It Yourself

1. Group by `Sub-Category` and find total `Profit`, sorted ascending (most loss-making first) — which sub-category loses the most money overall?
2. Build a pivot table showing total `Sales` with `Segment` as rows and `Ship Mode` as columns.
3. Using `.groupby()` with `.agg()`, compute both the average and the maximum `Shipping Days` per `Ship Mode`.

In [20]:
# TODO 1: total Profit per Sub-Category, sorted ascending


# TODO 2: pivot table -- Sales by Segment (rows) x Ship Mode (columns)


# TODO 3: average and max Shipping Days per Ship Mode

### Step 7 — Insight Synthesis

- **Profit by state**: California is the strongest performer (~$76.4K total profit); Texas is the biggest drag (~-$25.7K total loss) — worth checking whether that's driven by discounting practices specific to Texas.
- **Discount vs. Profit**: a clear, monotonic relationship. Average profit per order falls as the discount band rises — roughly +$67 with no discount, +$27 at Low, **-$78 at Medium, -$107 at High**. Discounts above ~20% are, on average, selling at a loss.
- **Ship Mode**: shipping time behaves exactly as expected (Same Day ≈ 0 days, Standard Class ≈ 5 days), but average profit per order is fairly flat across all four modes (~$28–32) — shipping choice doesn't appear to drive profitability on its own.
- **Seasonality**: Sales rise sharply toward year-end (November and December are the two highest months, roughly 3–4x February's total), consistent with holiday retail patterns. By weekday, Monday/Tuesday/Wednesday outsell Friday by close to 2x.
- **Customer value**: spend is concentrated — the top RFM customer alone accounts for ~$25K in lifetime Sales, well above the ~$2.9K average, suggesting a small set of high-value accounts worth treating differently from the broader base.

**Caveat carried from cleaning:** these Profit/Sales totals reflect the decision *not* to collapse the 7 legitimate `Order ID` + `Product ID` duplicate pairs — collapsing them would have understated every total above.

### Step 8 — Handoff Prep for Visualization

The next stage (lectures 7–8) turns these tables into charts. This notebook's job is to make sure nothing needs recomputing there — everything it needs already exists, by name, above.

In [21]:
# Tables and features the visualization stage will need -- named here so
# nothing has to be recomputed in the next notebook.
handoff_manifest = {
    "df_clean": "row-level cleaned + feature-engineered DataFrame",
    "customer_features": "one row per Customer ID -- spend, order count, tenure",
    "product_features": "one row per Product ID -- units sold, revenue, avg discount",
    "rfm": "one row per Customer ID -- Recency, Frequency, Monetary",
    "state_profit": "total Profit per State",
    "discount_profit": "average Profit per Discount Bucket",
    "ship_mode_summary": "average Profit and Shipping Days per Ship Mode",
    "region_summary": "total Sales and Profit per Region",
    "category_summary": "total Sales, avg Profit, order count per Category",
    "pivot": "average Sales by Region x Category",
}
for name, description in handoff_manifest.items():
    print(f"{name}: {description}")

df_clean: row-level cleaned + feature-engineered DataFrame
customer_features: one row per Customer ID -- spend, order count, tenure
product_features: one row per Product ID -- units sold, revenue, avg discount
rfm: one row per Customer ID -- Recency, Frequency, Monetary
state_profit: total Profit per State
discount_profit: average Profit per Discount Bucket
ship_mode_summary: average Profit and Shipping Days per Ship Mode
region_summary: total Sales and Profit per Region
category_summary: total Sales, avg Profit, order count per Category
pivot: average Sales by Region x Category


### Step 9 — Persist Handoff Tables to Disk

Listing the tables above isn't enough on its own -- `visualization_dashboard.ipynb` runs its **own kernel** and has no access to the variables sitting in this one's memory. Every object named in `handoff_manifest` gets written to a `handoff_data/` folder here, and the visualization notebook's Step 0 reads them straight back in.

- **Parquet** for the row-level / per-entity tables (`df_clean`, `customer_features`, `product_features`, `rfm`) -- preserves dtypes (`datetime64`, `category`) exactly like the clean-data handoff from the cleaning notebook.
- **CSV** for the small summary tables (`state_profit`, `discount_profit`, `ship_mode_summary`, `region_summary`, `category_summary`, `pivot`) -- they're already chart-ready and don't carry dtypes worth preserving via Parquet.

In [22]:
import os

export_dir = "handoff_data"
os.makedirs(export_dir, exist_ok=True)

# Row-level / per-entity tables -- Parquet preserves dtypes
df_clean.to_parquet(f"{export_dir}/df_clean.parquet", engine="pyarrow")
customer_features.to_parquet(f"{export_dir}/customer_features.parquet", engine="pyarrow")
product_features.to_parquet(f"{export_dir}/product_features.parquet", engine="pyarrow")
rfm.to_parquet(f"{export_dir}/rfm.parquet", engine="pyarrow")

# Small, chart-ready summary tables -- CSV is sufficient
state_profit.to_csv(f"{export_dir}/state_profit.csv")
discount_profit.to_csv(f"{export_dir}/discount_profit.csv")
ship_mode_summary.to_csv(f"{export_dir}/ship_mode_summary.csv")
region_summary.to_csv(f"{export_dir}/region_summary.csv")
category_summary.to_csv(f"{export_dir}/category_summary.csv")
pivot.to_csv(f"{export_dir}/pivot_region_category.csv")

print(f"Exported {len(os.listdir(export_dir))} files to '{export_dir}/':")
for filename in sorted(os.listdir(export_dir)):
    print(f"  {filename}")

Exported 10 files to 'handoff_data/':
  category_summary.csv
  customer_features.parquet
  df_clean.parquet
  discount_profit.csv
  pivot_region_category.csv
  product_features.parquet
  region_summary.csv
  rfm.parquet
  ship_mode_summary.csv
  state_profit.csv
